# Ejercicio 2: el 95% que no detecta nada

`pagos_taxi.csv`: 6000 pagos de la app de taxis, de septiembre y octubre de 2026. `es_fraude` vale 1 si el pago resultó ser fraudulento (tarjeta robada o contracargo). Mismo esqueleto del Ejercicio 1: `evaluar` corre una validación cruzada con el esquema que le pases y guarda una fila por modelo en `resultados`.

Los pasos 1 a 3 tienen `TODO`; los pasos 4 y 5 solo dicen qué variable dejar. La última celda imprime una comprobación por paso; no la modifiques. Corre todas las celdas una vez antes de empezar (**Run All**).

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pagos = pd.read_csv("pagos_taxi.csv")
pagos["hora"] = pd.to_datetime(pagos["fecha_hora"]).dt.hour
pagos["monto_por_km"] = pagos["monto"] / pagos["distancia_km"]

NUMERICAS = ["monto", "distancia_km", "monto_por_km", "hora", "tarjeta_nueva", "intentos_fallidos", "viajes_previos"]
CATEGORICAS = ["metodo"]
X = pagos[NUMERICAS + CATEGORICAS]
y = pagos["es_fraude"]
print("Pagos:", len(pagos), " fraudes:", y.sum(), f" ({y.mean():.1%})")

# La linea base nunca predice fraude: su precision no esta definida y
# scikit-learn avisa en cada pliegue. El 0 que reporta ya dice lo mismo.
warnings.filterwarnings("ignore", message="Precision is ill-defined")

Pagos: 6000  fraudes: 295  (4.9%)


## La función que evalúa cada modelo

In [2]:
resultados = {}


def preparacion():
    return ColumnTransformer([
        ("numericas", StandardScaler(), NUMERICAS),
        ("categoricas", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAS),
    ])


def evaluar(nombre, modelo, esquema):
    """Validacion cruzada del modelo con el esquema dado. Guarda en
    resultados el promedio de cada metrica entre los pliegues."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, scoring=["accuracy", "recall", "precision"])
    resultados[nombre] = {
        "modelo": nombre,
        "accuracy": round(puntajes["test_accuracy"].mean(), 4),
        "recall": round(puntajes["test_recall"].mean(), 4),
        "precision": round(puntajes["test_precision"].mean(), 4),
    }
    print(f"{nombre:20s} accuracy {resultados[nombre]['accuracy']:.4f}   recall {resultados[nombre]['recall']:.4f}   precision {resultados[nombre]['precision']:.4f}")

## Paso 1: un esquema que respete la proporción de fraudes

In [3]:
# TODO 1: el esquema de validacion. Con 5% de fraudes, cada pliegue tiene
# que tener la misma proporcion de fraudes que el total: usa el esquema
# estratificado de 5 pliegues, mezclado, con random_state=42.
esquema = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for numero, (_, indices_validacion) in enumerate(esquema.split(X, y)):
    print(f"Pliegue {numero + 1}: {y.iloc[indices_validacion].sum()} fraudes de {len(indices_validacion)}")

Pliegue 1: 59 fraudes de 1200
Pliegue 2: 59 fraudes de 1200
Pliegue 3: 59 fraudes de 1200
Pliegue 4: 59 fraudes de 1200
Pliegue 5: 59 fraudes de 1200


## Paso 2: la línea base

In [4]:
# TODO 2: la linea base, un clasificador que siempre predice la clase mas
# frecuente, con el nombre "linea base".
evaluar("linea base", DummyClassifier(strategy="most_frequent"), esquema)

linea base           accuracy 0.9508   recall 0.0000   precision 0.0000


## Paso 3: dos modelos

In [5]:
# TODO 3: una regresion logistica (max_iter=1000) con el nombre
# "logistica" y un gradient boosting (random_state=42) con el nombre
# "gradient boosting".
evaluar("logistica", LogisticRegression(max_iter=1000), esquema)
evaluar("gradient boosting", GradientBoostingClassifier(random_state=42), esquema)

logistica            accuracy 0.9613   recall 0.2576   precision 0.8652


gradient boosting    accuracy 0.9712   recall 0.4610   precision 0.9051


## Paso 4: la matriz de confusión

In [6]:
# Deja en probabilidades la probabilidad de fraude que el gradient boosting
# le asigna a cada pago cuando ese pago esta en el pliegue de validacion
# (cada pago recibe una sola, del modelo que no lo vio al entrenar), y en
# matriz la matriz de confusion de esas probabilidades con el umbral 0.5.
pipeline_gb = Pipeline([("preparar", preparacion()), ("modelo", GradientBoostingClassifier(random_state=42))])
probabilidades = cross_val_predict(pipeline_gb, X, y, cv=esquema, method="predict_proba")[:, 1]
matriz = confusion_matrix(y, probabilidades >= 0.5)
print(pd.DataFrame(matriz, index=["real legitimo", "real fraude"], columns=["marcado legitimo", "marcado fraude"]))

               marcado legitimo  marcado fraude
real legitimo              5691              14
real fraude                 159             136


## Paso 5: el umbral según el costo de cada error

In [7]:
# Cada fraude que pasa sin detectar le cuesta a la empresa 40 dolares (el
# contracargo, en promedio). Cada pago legitimo que se marca como fraude le
# cuesta 3 (la revision manual y el pasajero molesto). Calcula el costo
# total para cada umbral de UMBRALES, guardalo en costos (un diccionario
# umbral -> costo) y deja en umbral_elegido el de menor costo.
COSTO_FRAUDE_NO_DETECTADO = 40
COSTO_FALSA_ALARMA = 3
UMBRALES = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
costos = {}
for umbral in UMBRALES:
    tn, fp, fn, tp = confusion_matrix(y, probabilidades >= umbral).ravel()
    costos[umbral] = fn * COSTO_FRAUDE_NO_DETECTADO + fp * COSTO_FALSA_ALARMA
    print(f"umbral {umbral:.2f}: fraudes sin detectar {fn:3d}   falsas alarmas {fp:4d}   costo {costos[umbral]:6d} dolares")
umbral_elegido = min(costos, key=costos.get)
print("Umbral elegido:", umbral_elegido)

umbral 0.05: fraudes sin detectar 100   falsas alarmas  589   costo   5767 dolares
umbral 0.10: fraudes sin detectar 122   falsas alarmas  222   costo   5546 dolares
umbral 0.15: fraudes sin detectar 135   falsas alarmas  121   costo   5763 dolares
umbral 0.20: fraudes sin detectar 145   falsas alarmas   75   costo   6025 dolares
umbral 0.30: fraudes sin detectar 150   falsas alarmas   39   costo   6117 dolares
umbral 0.50: fraudes sin detectar 159   falsas alarmas   14   costo   6402 dolares
Umbral elegido: 0.1


## Comprobación

In [8]:
def comprobar(numero, condicion):
    if condicion:
        print(f"Paso {numero}: OK")
    else:
        print(f"Paso {numero}: revisa")


def fila(nombre):
    return resultados.get(nombre)


comprobar(1, isinstance(esquema, StratifiedKFold) and esquema.shuffle and esquema.random_state == 42)
comprobar(2, fila("linea base") is not None and fila("linea base")["accuracy"] > 0.95 and fila("linea base")["recall"] == 0)
comprobar(3, fila("logistica") is not None and fila("gradient boosting") is not None and fila("gradient boosting")["recall"] > fila("logistica")["recall"])
comprobar(4, matriz is not None and matriz.sum() == len(pagos) and probabilidades is not None and len(probabilidades) == len(pagos))
comprobar(5, len(costos) == len(UMBRALES) and umbral_elegido == 0.1)

Paso 1: OK
Paso 2: OK
Paso 3: OK
Paso 4: OK
Paso 5: OK


## Si terminas antes

La línea base tiene más accuracy que una moneda al aire y no detecta ni un fraude. ¿Qué número de la tabla de `resultados` lo delata, y por qué la accuracy no puede?

In [9]:
# Si la revision de una falsa alarma costara 20 dolares en vez de 3.
costos_20 = {}
for umbral in UMBRALES:
    tn, fp, fn, tp = confusion_matrix(y, probabilidades >= umbral).ravel()
    costos_20[umbral] = fn * COSTO_FRAUDE_NO_DETECTADO + fp * 20
    print(f"umbral {umbral:.2f}: fraudes sin detectar {fn:3d}   falsas alarmas {fp:4d}   costo {costos_20[umbral]:6d} dolares")
print("Umbral elegido con falsas alarmas de 20 dolares:", min(costos_20, key=costos_20.get))

umbral 0.05: fraudes sin detectar 100   falsas alarmas  589   costo  15780 dolares
umbral 0.10: fraudes sin detectar 122   falsas alarmas  222   costo   9320 dolares
umbral 0.15: fraudes sin detectar 135   falsas alarmas  121   costo   7820 dolares
umbral 0.20: fraudes sin detectar 145   falsas alarmas   75   costo   7300 dolares
umbral 0.30: fraudes sin detectar 150   falsas alarmas   39   costo   6780 dolares
umbral 0.50: fraudes sin detectar 159   falsas alarmas   14   costo   6640 dolares
Umbral elegido con falsas alarmas de 20 dolares: 0.5
